# Scam detector evaluation

This notebook shows the EMSCAD tuning metrics. The evaluation logic is in
`src/evaluation/suites/emscad.py`. The notebook calls that logic through
`python main.py eval emscad --part tuning`.

Put `fake_job_postings.csv` in `data/kaggle/` after you check the license.
The notebook does not download the file.

The held-out part is not in this notebook. It runs one time through the CLI.

## Setup

In [ ]:
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
BACKEND_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
CSV_PATH = BACKEND_DIR / "data" / "kaggle" / "fake_job_postings.csv"
if not CSV_PATH.is_file():
    raise SystemExit(
        f"CSV not found at {CSV_PATH}. "
        "Download it by hand after the license check and put it there."
    )
print(f"Backend: {BACKEND_DIR}")
print(f"CSV: {CSV_PATH}")

## Run the tuning part

In [ ]:
import subprocess
import sys

command = [
    sys.executable,
    "main.py",
    "eval",
    "emscad",
    "--part",
    "tuning",
    "--out",
    "data/eval/notebook",
]
print(" ".join(command))
completed = subprocess.run(command, cwd=BACKEND_DIR, check=True)
print(f"exit={completed.returncode}")

## Load the newest report

In [ ]:
import json

report_dir = BACKEND_DIR / "data" / "eval" / "notebook" / "emscad"
reports = sorted(report_dir.glob("*.json"))
if not reports:
    raise SystemExit(f"No report in {report_dir}")
report_path = reports[-1]
report = json.loads(report_path.read_text(encoding="utf-8"))
print(report_path)
print(f"cases={report['metadata']['case_count']}")

## Metrics and selection

In [ ]:
metrics = report["metrics"]
selection = report["details"]["selection"]
keys = [
    "precision",
    "precision.ci95_low",
    "precision.ci95_high",
    "recall",
    "recall.ci95_low",
    "recall.ci95_high",
    "false_positive_rate",
    "false_positive_rate.ci95_low",
    "false_positive_rate.ci95_high",
]
for key in keys:
    print(f"{key}: {metrics.get(key)}")
print("selection:", selection)

## Pipeline sweep

In [ ]:
import matplotlib.pyplot as plt

sweep = report["details"]["pipeline_sweep"]
thresholds = [row["threshold"] for row in sweep]
precision = [row["precision"] for row in sweep]
recall = [row["recall"] for row in sweep]
fpr = [row["false_positive_rate"] for row in sweep]
star = selection["threshold"]

figure, axis = plt.subplots(figsize=(8, 4))
axis.plot(thresholds, precision, label="precision")
axis.plot(thresholds, recall, label="recall")
axis.plot(thresholds, fpr, label="false-positive rate")
axis.axvline(star, color="black", linestyle="--", label=f"t*={star}")
axis.set_xlabel("threshold")
axis.set_ylabel("rate")
axis.set_ylim(0, 1.05)
axis.legend()
axis.set_title("Pipeline sweep on the tuning part")
plt.show()

## Confusion counts at t*

In [ ]:
star = selection["threshold"]
row = next(item for item in sweep if item["threshold"] == star)
print(row)
print("confusion at default:", report["details"]["confusion"])

## Detector reason counts by class

In [ ]:
from collections import Counter

by_class = {"scam": Counter(), "legit": Counter()}
for case in report["details"]["cases"]:
    label = case["expected"]
    for reason in case.get("detector_reasons") or []:
        by_class[label][reason] += 1
for label in ("scam", "legit"):
    print(label, sum(by_class[label].values()), "reason hits")
    for reason, count in by_class[label].most_common():
        print(f"  {count}  {reason}")

## Summary

The numbers above are the pipeline decision on the EMSCAD tuning part:
the quick filter, or a detector score at or above the threshold.

`t*` and the action come from fixed rules (false-positive rate at most
0.005; `drop` only with precision at least 0.50 and at least 10 flagged
frauds). When the action is `flag`, the pipeline keeps every posting and
records the score.

EMSCAD limits: postings from 2012 to 2014, mostly from the US; emails,
links, and phone numbers are masked; there are no company names and no
line breaks.

The held-out part runs one time, through the CLI, and never from this
notebook. Do not commit notebook outputs.